# SSIF 01: Longitudinal Academic Retention Panel Audit
### Student Success Intelligence Framework (SSIF)

This notebook executes an automated empirical audit of the 79,239-row longitudinal academic panel (`academic_survival_longitudinal.csv`):
- **Cohort:** 20,000 unique students tracked across semesters 1 through 8
- **Integrity Verifications:** Schema validation, data types, missingness mechanisms (MAR vs MCAR), and data leakage prevention (RULE-009, RULE-010).

In [ ]:
import sys
from pathlib import Path
# Add project root to path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_retention
from src.validation.schema_validator import validate_retention_schema
from src.validation.leakage_detector import check_retention_leakage
from src.validation.missingness_analyzer import analyze_missingness
from src.validation.data_profiler import profile_dataframe

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

## 1. Load & Validate Dataset Schema

In [ ]:
df = load_retention()
print(f"Dataset Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Unique Students: {df['Student_ID'].nunique():,}")
print(f"Observed Semesters: {sorted(df['Semester'].unique())}")
df.head()

## 2. Target Variable & Outcome Distribution

In [ ]:
print("--- Primary Classification Target (Target_Dropout_Next_Sem) ---")
target_counts = df["Target_Dropout_Next_Sem"].value_counts(dropna=False)
target_pcts = df["Target_Dropout_Next_Sem"].value_counts(normalize=True) * 100
for val, count in target_counts.items():
    print(f"Class {val}: {count:,} ({target_pcts[val]:.2f}%)")

print("\n--- End of Semester Realized Status ---")
status_counts = df["End_of_Semester_Status"].value_counts(dropna=False)
print(status_counts)

## 3. Data Leakage & Target Contamination Audit (RULE-009, RULE-010)

In [ ]:
leakage_report = check_retention_leakage(df)
print(f"Critical Leakage Detected: {leakage_report.critical_leakage}")
print("\nAudit Summary:")
for rec in leakage_report.checks:
    print(f"- [{rec.status}] {rec.check_name}: {rec.details}")

## 4. Missingness Analysis (MAR vs MCAR)

In [ ]:
missing_res = analyze_missingness(df, dataset_name="Retention Panel")
print(f"Total Columns with Missingness: {len(missing_res.columns_with_missing)}")
for col, pct in missing_res.missingness_percentages.items():
    if pct > 0:
        print(f"  {col}: {pct:.2f}% missing")

## 5. Visual Distribution of Academic Predictors by Dropout Outcome

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

sns.histplot(data=df, x="Sem_GPA", hue="Target_Dropout_Next_Sem", kde=True, ax=axes[0], palette="Set1", common_norm=False)
axes[0].set_title("Semester GPA by Next-Semester Dropout")

sns.histplot(data=df, x="Attendance", hue="Target_Dropout_Next_Sem", kde=True, ax=axes[1], palette="Set1", common_norm=False)
axes[1].set_title("Attendance Rate (%) by Next-Semester Dropout")

sns.boxplot(data=df, x="Target_Dropout_Next_Sem", y="Financial_Stress", ax=axes[2], palette="Set2")
axes[2].set_title("Financial Stress (1-5) by Dropout Outcome")

plt.tight_layout()
plt.show()